# Run 27 (Kaggle, T4 ×2) — Count-aware: B2 @256 RGB, no-mask aug + 3-fold + TTA + pseudo-labeling

From-scratch EfficientNet-B2-scale (**7.83M**, 23 MBConv blocks) at **256×256 RGB**, compound-scaled
up from the Run 24 config that broke the plateau (66 → 70.45% val). Competition-legal: PyTorch
primitives only, no pretrained weights.

**Kaggle setup:**
- Add the **signal-object-detection** competition as an *Input* (right panel → Add Input).
- Accelerator: **GPU T4 ×2** — the notebook auto-uses both via `nn.DataParallel` and scales the batch.
- `submission.csv` is written to `/kaggle/working/` (Output tab / Submit).

Recipe: dropout 0.4, drop_path 0.2, translation aug, EMA 0.999, OneCycleLR, 70 epochs. `N_FOLDS=1` to
calibrate (confirm > 70.45), then 3 + TTA for the submission.

In [ ]:
# Kaggle provides torch + the competition data — nothing to install.

In [ ]:
# === Device (T4 x2) ===
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = (device.type == 'cuda')
NGPU = torch.cuda.device_count()
print('torch', torch.__version__, '| device:', device, '| GPUs:', NGPU,
      '|', [torch.cuda.get_device_name(i) for i in range(NGPU)] if use_amp else 'CPU')

In [ ]:
# === Locate the competition data under /kaggle/input (robust + diagnostic) ===
import glob
from pathlib import Path
print('Contents of /kaggle/input:')
for p in sorted(glob.glob('/kaggle/input/*')):
    print('  -', p)
    for q in sorted(glob.glob(p+'/*'))[:10]:
        print('       ', Path(q).name + ('/' if Path(q).is_dir() else ''))

hits = sorted(glob.glob('/kaggle/input/**/train.csv', recursive=True), key=len)
assert hits, ('No train.csv under /kaggle/input. Right panel -> "+ Add Input" -> Competitions tab -> '
              'search "signal object detection" -> add it, then re-run.')
DATA_DIR = Path(hits[0]).parent

def imgdir(stem):
    d = DATA_DIR/stem
    if d.is_dir(): return d
    cands=[Path(x) for x in glob.glob(str(DATA_DIR)+'/**/'+stem, recursive=True) if Path(x).is_dir()]
    return cands[0] if cands else d
TRAIN_DIR, TEST_DIR = imgdir('train'), imgdir('test')
print('\nDATA_DIR  =', DATA_DIR)
print('TRAIN_DIR =', TRAIN_DIR, '| #files:', len(glob.glob(str(TRAIN_DIR)+'/*')))
print('TEST_DIR  =', TEST_DIR,  '| #files:', len(glob.glob(str(TEST_DIR)+'/*')))
assert TRAIN_DIR.is_dir() and TEST_DIR.is_dir(), 'Found train.csv but not train/ & test/ folders — check the listing above and set TRAIN_DIR/TEST_DIR manually.'

In [ ]:
# === Data: compute RGB stats + cache all images in RAM as uint8 (no DataLoader) ===
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as F
from PIL import Image

IMG = 256
train_df = pd.read_csv(DATA_DIR/'train.csv')
test_df  = pd.read_csv(DATA_DIR/'test.csv')

# per-channel mean/std from a sample (RGB spectrograms != ImageNet stats)
_smp = train_df['id'].sample(800, random_state=0)
_acc=np.zeros(3); _acc2=np.zeros(3); _n=0
for fn in _smp:
    a=np.asarray(Image.open(TRAIN_DIR/fn).convert('RGB').resize((IMG,IMG)),np.float32)/255.
    _acc+=a.reshape(-1,3).sum(0); _acc2+=(a.reshape(-1,3)**2).sum(0); _n+=IMG*IMG
MEAN=torch.tensor(_acc/_n,dtype=torch.float32).view(3,1,1)
STD =torch.tensor(np.sqrt(_acc2/_n-(_acc/_n)**2),dtype=torch.float32).view(3,1,1)
print('RGB mean',MEAN.flatten().tolist(),'std',STD.flatten().tolist())

def cache_u8(df,img_dir):
    X=torch.empty(len(df),3,IMG,IMG,dtype=torch.uint8)
    for i,row in enumerate(df.itertuples(index=False)):
        a=np.asarray(Image.open(img_dir/row.id).convert('RGB').resize((IMG,IMG)))
        X[i]=torch.from_numpy(a.copy()).permute(2,0,1)
        if (i+1)%3000==0: print(f'  {i+1}/{len(df)}',flush=True)
    return X
print('caching train...'); XTR=cache_u8(train_df,TRAIN_DIR)
print('caching test...');  XTE=cache_u8(test_df, TEST_DIR)
YTR=torch.tensor(train_df['label'].values-1)
print('train',tuple(XTR.shape),f'{XTR.nelement()/1e9:.1f}GB uint8 | test',tuple(XTE.shape))

In [ ]:
# === 5. Model — from-scratch EfficientNet/MBConv (PyTorch primitives only) ===
class DropPath(nn.Module):
    def __init__(s,p=0.0): super().__init__(); s.p=p
    def forward(s,x):
        if s.p==0.0 or not s.training: return x
        k=1-s.p; m=torch.empty((x.size(0),1,1,1),dtype=x.dtype,device=x.device).bernoulli_(k); return x/k*m

class MBConv(nn.Module):
    def __init__(s,cin,cout,k=3,st=1,t=6,se=0.25,dp=0.0):
        super().__init__()
        cm=cin*t; s.use_res=(st==1 and cin==cout); L=[]
        if t!=1: L+=[nn.Conv2d(cin,cm,1,bias=False),nn.BatchNorm2d(cm),nn.SiLU(True)]
        L+=[nn.Conv2d(cm,cm,k,st,k//2,groups=cm,bias=False),nn.BatchNorm2d(cm),nn.SiLU(True)]
        s.conv=nn.Sequential(*L); sc=max(1,int(cin*se))
        s.se=nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(cm,sc,1),nn.SiLU(True),nn.Conv2d(sc,cm,1),nn.Sigmoid())
        s.proj=nn.Sequential(nn.Conv2d(cm,cout,1,bias=False),nn.BatchNorm2d(cout)); s.dp=DropPath(dp)
        if s.use_res: nn.init.zeros_(s.proj[1].weight)
    def forward(s,x):
        o=s.conv(x); o=o*s.se(o); o=s.proj(o); return x+s.dp(o) if s.use_res else o

class EffNet(nn.Module):
    # (expand t, out_ch, kernel, stride, repeats) — EfficientNet-B0 layout
    cfg=[(1,16,3,1,1),(6,24,3,2,2),(6,40,5,2,2),(6,80,3,2,3),(6,112,5,1,3),(6,192,5,2,4),(6,320,3,1,1)]
    def __init__(s,num_classes=5,dropout=0.3,drop_path=0.1,width=1.0,depth_mult=1.0):
        super().__init__()
        import math
        ch=lambda c:int(c*width)
        rep=lambda r:int(math.ceil(r*depth_mult))
        s.stem=nn.Sequential(nn.Conv2d(3,ch(32),3,2,1,bias=False),nn.BatchNorm2d(ch(32)),nn.SiLU(True))
        blocks=[]; cin=ch(32); tot=sum(rep(r) for *_,r in s.cfg); bi=0
        for t,co,k,st,r in s.cfg:
            co=ch(co)
            for j in range(rep(r)):
                blocks.append(MBConv(cin,co,k,st if j==0 else 1,t,dp=drop_path*bi/max(1,tot-1))); cin=co; bi+=1
        s.blocks=nn.Sequential(*blocks)
        s.head=nn.Sequential(nn.Conv2d(cin,ch(1280),1,bias=False),nn.BatchNorm2d(ch(1280)),nn.SiLU(True),
                             nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Dropout(dropout),nn.Linear(ch(1280),num_classes))
        for m in s.modules():
            if isinstance(m,nn.Conv2d) and m.groups==1: nn.init.kaiming_normal_(m.weight,mode='fan_out',nonlinearity='relu')
            elif isinstance(m,nn.Linear): nn.init.zeros_(m.bias) if m.bias is not None else None
    def forward(s,x): return s.head(s.blocks(s.stem(x)))

_m=EffNet(width=1.1,depth_mult=1.2); _n=sum(p.numel() for p in _m.parameters()); print(f'EffNet B2-scale params: {_n:,} ({_n/1e6:.2f}M)')
print('out', tuple(_m(torch.randn(2,3,IMG,IMG)).shape))

In [ ]:
# === 6. EMA ===
import copy
class EMA:
    def __init__(s,m,d=0.999): s.d=d; s.sh=copy.deepcopy(m).eval(); [p.requires_grad_(False) for p in s.sh.parameters()]
    @torch.no_grad()
    def update(s,m):
        for a,b in zip(s.sh.state_dict().values(),m.state_dict().values()):
            if a.dtype.is_floating_point: a.mul_(s.d).add_(b,alpha=1-s.d)
            else: a.copy_(b)

In [ ]:
# === Train one fold: count-preserving GPU aug + optional pseudo-labels (Run 27) ===
CONFIG = dict(epochs=70, batch_size=48, max_lr=1.0e-3, weight_decay=1e-3,
              label_smoothing=0.1, dropout=0.4, drop_path=0.2, ema_decay=0.999,
              width=1.1, depth_mult=1.2,                # ~EfficientNet-B2 scale (proven Run 25 reg)
              ema_reset_epoch=3, pct_start=0.2)

from sklearn.model_selection import StratifiedKFold
MEAN_d = MEAN.to(device); STD_d = STD.to(device)

def gpu_aug(x):     # COUNT-PRESERVING only: translation + contrast + SNR-varying noise.
    # NO masking/cutout — Exp 35 showed masking erases objects and costs -3.8pp on this counting task.
    fs=int(torch.randint(-18,19,(1,)).item()); ts=int(torch.randint(-28,29,(1,)).item())
    x=torch.roll(x,shifts=(fs,ts),dims=(2,3))
    if fs>0: x[:,:,:fs,:]=0
    elif fs<0: x[:,:,fs:,:]=0
    if ts>0: x[:,:,:,:ts]=0
    elif ts<0: x[:,:,:,ts:]=0
    B=x.size(0)
    c=0.8+0.4*torch.rand(B,1,1,1,device=x.device); b=(torch.rand(B,1,1,1,device=x.device)-0.5)*0.1
    sig=0.01+0.05*torch.rand(B,1,1,1,device=x.device)          # SNR-varying additive noise
    x=(x-0.5)*c+0.5+b + torch.randn_like(x)*sig
    return x.clamp_(0,1)

@torch.no_grad()
def evaluate_cache(model, idx):     # always evaluates on REAL train cache -> leakage-free CV
    model.eval(); correct=0
    with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=use_amp):
        for s0 in range(0,len(idx),256):
            bi=idx[s0:s0+256]
            x=(XTR[bi].to(device).float()/255.-MEAN_d)/STD_d
            correct+=(model(x).float().argmax(1).cpu()==YTR[bi]).sum().item()
    return 100*correct/len(idx)

def train_fold(tr_idx, va_idx, cfg, tag='fold0', seed=42, xext=None, yext=None, pw=0.5):
    # xext/yext: optional pseudo-labeled TEST images (uint8 [M,3,H,W], labels [M]) mixed into TRAIN at weight pw.
    torch.manual_seed(seed)
    bs=cfg['batch_size']*max(1,NGPU)
    core=EffNet(5,cfg['dropout'],cfg['drop_path'],width=cfg['width'],depth_mult=cfg['depth_mult']).to(device)
    model=nn.DataParallel(core) if NGPU>1 else core
    ema=EMA(core,cfg['ema_decay']); crit=nn.CrossEntropyLoss(label_smoothing=cfg['label_smoothing'],reduction='none')
    opt=torch.optim.AdamW(model.parameters(),lr=cfg['max_lr'],weight_decay=cfg['weight_decay'])
    scaler=torch.amp.GradScaler('cuda',enabled=use_amp)
    tr_idx=torch.as_tensor(tr_idx); va_idx=torch.as_tensor(va_idx)
    # combined training pool: real (weight 1.0) + optional pseudo (weight pw)
    Xf=XTR[tr_idx]; yf=YTR[tr_idx]; wf=torch.ones(len(tr_idx))
    npseudo=0
    if xext is not None and len(xext)>0:
        npseudo=len(xext); Xf=torch.cat([Xf,xext]); yf=torch.cat([yf,yext]); wf=torch.cat([wf,torch.full((npseudo,),pw)])
    steps=len(Xf)//bs
    sched=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=cfg['max_lr'],total_steps=cfg['epochs']*steps,pct_start=cfg['pct_start'])
    print(f'[{tag}] {len(tr_idx)} real + {npseudo} pseudo / {len(va_idx)} val | {steps} batches/ep | {NGPU} GPU, batch {bs}',flush=True)
    best=0.0; best_state=None
    for ep in range(cfg['epochs']):
        if ep==cfg['ema_reset_epoch']: ema=EMA(core,cfg['ema_decay'])
        model.train(); tc=tn=0
        perm=torch.randperm(len(Xf))
        for s0 in range(0,steps*bs,bs):
            bi=perm[s0:s0+bs]
            x=(gpu_aug(Xf[bi].to(device,non_blocking=True).float()/255.)-MEAN_d)/STD_d
            yb=yf[bi].to(device,non_blocking=True); wb=wf[bi].to(device,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=use_amp):
                out=model(x); loss=(crit(out,yb)*wb).sum()/wb.sum()
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); ema.update(core)
            tc+=(out.float().argmax(1)==yb).sum().item(); tn+=yb.size(0)
        ta=100*tc/tn; rv=evaluate_cache(core,va_idx); ev=evaluate_cache(ema.sh,va_idx)
        acc=max(rv,ev); s='EMA' if ev>=rv else 'raw'
        print(f'[{tag}] ep{ep+1}/{cfg["epochs"]} train {ta:.2f} raw {rv:.2f} ema {ev:.2f} -> val {acc:.2f} ({s}) gap {ta-acc:+.1f} lr {opt.param_groups[0]["lr"]:.2e}',flush=True)
        if acc>best:
            best=acc; chosen=ema.sh if ev>=rv else core
            best_state={k:v.detach().cpu().clone() for k,v in chosen.state_dict().items()}
            torch.save(best_state,f'/kaggle/working/best_{tag}.pt')
    print(f'[{tag}] BEST {best:.2f}%'); return best,best_state

In [ ]:
# === 3-fold ensemble + semi-supervised pseudo-labeling, with leakage-free CV gate (Run 27) ===
N_FOLDS = 3
TAU = 0.95      # pseudo-label confidence threshold (keep HIGH)
PW  = 0.5       # pseudo-label loss weight vs real
N_ROUNDS = 1    # pseudo-label rounds (set 0 to skip pseudo and just do the 3-fold ensemble)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
splits = list(skf.split(np.arange(len(train_df)), train_df['label']))

def predict_test_probs(states):                 # fold-averaged softmax over test (no TTA -> honest confidence)
    probs = torch.zeros(len(test_df), 5)
    for st in states:
        m = EffNet(5, CONFIG['dropout'], CONFIG['drop_path'], width=CONFIG['width'], depth_mult=CONFIG['depth_mult']).to(device)
        m.load_state_dict(st); m.eval()
        with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            for s0 in range(0, len(test_df), 256):
                x = (XTE[s0:s0+256].to(device).float()/255. - MEAN_d)/STD_d
                probs[s0:s0+256] += torch.softmax(m(x).float(), 1).cpu()
    return probs/len(states)

def build_pseudo(probs):                        # high-confidence, class-capped (prevents count drift)
    conf, lab = probs.max(1); idx = torch.where(conf > TAU)[0]
    per = [int((lab[idx]==c).sum()) for c in range(5)]; cap = int(np.median([p for p in per if p>0]) or 0)
    sel = []
    for c in range(5):
        ci = idx[lab[idx]==c]; sel.append(ci[conf[ci].argsort(descending=True)][:cap])
    sel = torch.cat(sel) if len(sel) else torch.tensor([], dtype=torch.long)
    print(f'pseudo: {len(idx)}/{len(test_df)} conf>{TAU}, capped {cap}/class -> {len(sel)} used; '
          f'dist {[int((lab[sel]==c).sum()) for c in range(5)]}', flush=True)
    return sel, lab[sel]

def run_round(xext, yext, tagp):
    states, accs = [], []
    for f,(tri,vai) in enumerate(splits):
        b, st = train_fold(tri, vai, CONFIG, tag=f'{tagp}f{f}', seed=42+f, xext=xext, yext=yext, pw=PW)
        accs.append(b); states.append(st)
    cv = float(np.mean(accs)); print(f'[{tagp}] CV {cv:.2f}%  folds {[round(a,1) for a in accs]}', flush=True)
    return cv, states

cv0, fold_states = run_round(None, None, 'R0')          # round 0: real labels only
best_cv = cv0
for r in range(1, N_ROUNDS+1):
    sel, plab = build_pseudo(predict_test_probs(fold_states))
    cv, states = run_round(XTE[sel], plab, f'R{r}')     # retrain from scratch w/ test pseudo-labels
    if cv >= best_cv + 0.5:                              # GATE: clean-fold CV must improve
        print(f'R{r} PROMOTED ({cv:.2f} >= {best_cv:.2f}+0.5)'); best_cv = cv; fold_states = states
    else:
        print(f'R{r} rejected ({cv:.2f} < {best_cv:.2f}+0.5) -> keep previous round'); break
print(f'\nFINAL CV {best_cv:.2f}%  (Kaggle est ~{best_cv+2.7:.1f}%)')

In [ ]:
# === Inference from cached test tensors -> submission.csv ===
probs=torch.zeros(len(test_df),5)
for st in fold_states:
    m=EffNet(5,CONFIG['dropout'],CONFIG['drop_path'],width=CONFIG['width'],depth_mult=CONFIG['depth_mult']).to(device)
    m.load_state_dict(st); m.eval()
    with torch.no_grad(), torch.autocast(device_type=device.type,dtype=torch.float16,enabled=use_amp):
        TTA=[(0,0),(0,12),(0,-12),(6,0),(-6,0)]   # Run 26: translation TTA (valid; no flips/rot)
        for s0 in range(0,len(test_df),256):
            x0=(XTE[s0:s0+256].to(device).float()/255.-MEAN_d)/STD_d
            for fs,ts in TTA:
                x=torch.roll(x0,shifts=(fs,ts),dims=(2,3))
                probs[s0:s0+256]+=torch.softmax(m(x).float(),1).cpu()
pred=probs.argmax(1).numpy()+1
sub=pd.DataFrame({'id':test_df['id'].tolist(),'label':pred}); sub.to_csv('/kaggle/working/submission.csv',index=False)
print(sub['label'].value_counts().sort_index()); print('-> /kaggle/working/submission.csv')